In [31]:
import os
import json
import pathlib
import logging
from string import Template
from dotenv import load_dotenv
from openai import OpenAI
from typing import List, Dict


In [32]:
BASE_DIR = pathlib.Path.cwd()

load_dotenv(BASE_DIR / "environment.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY not set in .env")



In [33]:
client = OpenAI(api_key=OPENAI_API_KEY)


In [34]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s"
)
logger = logging.getLogger("pipeline")


In [35]:
with open(BASE_DIR / "config.json", "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

OPENAI_CFG = CONFIG["openai"]
OUTPUT_CFG = CONFIG["output"]


In [36]:
OUTPUT_DIR = BASE_DIR / OUTPUT_CFG.get("output_dir")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [37]:
with open(BASE_DIR / "toulmin_prompt.txt", "r", encoding="utf-8") as f:
    TOULMIN_TEMPLATE = Template(f.read())

In [38]:
def build_articles_block(articles: List[Dict]) -> str:
    """Format multiple abstracts into a single block for the LLM prompt.

    Args:
        articles: List of article dicts with metadata and abstracts.

    Returns:
        Formatted string block containing article details.
    """
    blocks = []
    for idx, art in enumerate(articles, start=1):
        lines = [
            f"Article {idx}:",
            f"PMID: {art.get('pmid') or art.get('pubmed_id', '')}",
            f"Evidence type: {art.get('_evidence_type', '')}",
            f"Year: {art.get('year', '')}",
            f"Title: {art.get('title', '')}",
            "Abstract:",
            art.get("abstract", "") or ""
        ]
        blocks.append("\n".join(lines))
    return "\n\n".join(blocks)


def build_toulmin_prompt(claim_text: str, articles: List[Dict]) -> str:
    """Build the LLM prompt for Toulmin extraction.

    Args:
        claim_text: The claim text to evaluate.
        articles: Retrieved articles to include in the prompt.

    Returns:
        Prompt string for the LLM.
    """
    articles_block = build_articles_block(articles)
    return TOULMIN_TEMPLATE.substitute(
        claim_text=claim_text,
        articles_block=articles_block
    )


In [39]:
def extract_toulmin_argument(articles: List[Dict], claim_text: str) -> Dict:
    """Extract Toulmin argument structure for a claim using the LLM.

    Args:
        articles: Retrieved articles used as evidence.
        claim_text: The claim text to evaluate.

    Returns:
        Dict containing stance and Toulmin fields, or a parse/error fallback.
    """
    if not articles:
        return {
            "stance": "irrelevant",
            "toulmin": {
                "claim": "",
                "data": [],
                "warrant": "",
                "backing": [],
                "qualifier": "",
                "rebuttals": []
            }
        }

    try:
        prompt = build_toulmin_prompt(claim_text, articles)
    except Exception as exc:
        logger.exception("Failed to build Toulmin prompt")
        return {
            "stance": "prompt_error",
            "toulmin": {
                "claim": "",
                "data": [],
                "warrant": "",
                "backing": [],
                "qualifier": "",
                "rebuttals": [],
                "error": str(exc)
            }
        }

    try:
        response = client.chat.completions.create(
            model=OPENAI_CFG["model"],
            temperature=OPENAI_CFG.get("temperature", 0),
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "You are a precise biomedical argument extraction assistant."},
                {"role": "user", "content": prompt}
            ]
        )
    except Exception as exc:
        logger.exception("LLM request failed")
        return {
            "stance": "llm_error",
            "toulmin": {
                "claim": "",
                "data": [],
                "warrant": "",
                "backing": [],
                "qualifier": "",
                "rebuttals": [],
                "error": str(exc)
            }
        }

    content = response.choices[0].message.content
    try:
        parsed = json.loads(content)
    except json.JSONDecodeError:
        logger.warning("Failed to parse LLM response as JSON")
        parsed = {
            "stance": "parse_error",
            "toulmin": {
                "claim": "",
                "data": [],
                "warrant": "",
                "backing": [],
                "qualifier": "",
                "rebuttals": [],
                "raw": content
            }
        }

    return parsed


In [40]:
RETRIEVAL_RESULTS_PATH = pathlib.Path(
    CONFIG.get("retrieval", {}).get("results_path", BASE_DIR / "retrieval_outputs/retrieval_results.jsonl")
)


In [41]:
def run_pipeline():
    """Run the Toulmin extraction pipeline over retrieval results.

    Args:
        None

    Returns:
        None
    """
    if not RETRIEVAL_RESULTS_PATH.exists():
        logger.error("Missing retrieval results: %s", RETRIEVAL_RESULTS_PATH)
        return

    try:
        with RETRIEVAL_RESULTS_PATH.open("r", encoding="utf-8") as f:
            lines = [ln.strip() for ln in f if ln.strip()]
    except Exception:
        logger.exception("Failed to read retrieval results")
        return

    if not lines:
        logger.warning("No retrieval records found.")
        return

    for line in lines:
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            logger.warning("Skipping invalid JSON line in retrieval results")
            continue

        claim_id = rec.get("claim_id", "retrieval")
        claim_text = rec.get("query", "")
        articles = rec.get("articles", [])

        logger.info("Processing retrieval record %s", claim_id)
        logger.info("Query: %s", claim_text)
        logger.info("Total articles to process with LLM: %s", len(articles))

        toulmin_arg = extract_toulmin_argument(articles, claim_text)

        sources = [
            {
                "pubmed_id": art.get("pubmed_id") or art.get("pmid"),
                "title": art.get("title"),
                "year": art.get("year"),
                "journal": art.get("journal"),
                "evidence_type": art.get("_evidence_type"),
                "abstract": art.get("abstract")
            }
            for art in articles
        ]

        record = {
            "claim_id": claim_id,
            "claim_text": claim_text,
            "sources": sources,
            "stance": toulmin_arg.get("stance"),
            "toulmin": toulmin_arg.get("toulmin", {})
        }

        out_path = OUTPUT_DIR / f"{claim_id}_toulmin.jsonl"
        try:
            with out_path.open("w", encoding="utf-8") as f_out:
                f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
        except Exception:
            logger.exception("Failed to write output for %s", claim_id)
            continue

        logger.info("Saved Toulmin arguments for %s to %s", claim_id, out_path)


In [42]:
run_pipeline()

2026-03-01 14:03:44,808 INFO pipeline: Processing retrieval record retrieval
2026-03-01 14:03:44,808 INFO pipeline: Query: Semaglutide induces significant weight loss in adults with obesity.
2026-03-01 14:03:44,808 INFO pipeline: Total articles to process with LLM: 15
2026-03-01 14:04:03,506 INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-01 14:04:03,517 INFO pipeline: Saved Toulmin arguments for retrieval to /Users/ftzavellos/Law_and_Tech/drug_explanations/Code/outputs/retrieval_toulmin.jsonl
